# 08 — Run the promoted full-match pipeline

## Problem

The final CSV must contain one label for every tactical frame, including long missing-ball intervals. The public five-minute labels are used only by the evaluator, never while constructing the submission.

## Idea

Use the conservative three-family hypothesis resolver: live handoffs (`H ↔ A`), stoppage onset (`H/A → D`), and restart activation (`D → H/A`). A dead-ball hypothesis must agree with forward settling evidence before `ENTER_D`; restart activation or persistent control releases it. Ambiguous evidence remains `CONTESTED` rather than forcing a switch.

## Construction

The modular runner calls raw data loading, temporal and missing-ball features, scenarios, anchors, transition hypotheses, the bounded resolver, and the final `H/A/D/X` writer. Optional alternate-camera observations are documented in the appendix and are not silently inserted into this run.

```mermaid
flowchart LR
R[Raw tracking + Smart Tags] --> F[Physical and missing-ball features]
F --> S[Football scenarios]
S --> A[Conservative anchors]
A --> H[Transition hypotheses]
H --> Q[Bounded interval resolver]
Q --> O[Final H/A/D/X CSV]
```

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'possession_lab').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA_ROOT = Path(os.environ.get('MTAH_DATA_DIR', ROOT / 'data'))
assert (DATA_ROOT / 'tactical').exists(), f'Missing tactical data: {DATA_ROOT}'
os.environ['MTAH_DATA_DIR'] = str(DATA_ROOT)
print('project root:', ROOT)
print('data root:', DATA_ROOT)

## Execute the promoted model

This cell starts from the supplied raw data and writes one frame-aligned CSV. It does not load the public ground truth.

In [ ]:
from possession_lab.reference_pipeline import run_reference_pipeline

final_csv = run_reference_pipeline(
    data_dir=DATA_ROOT,
    output_path=ROOT / 'evaluation' / 'results' / 'final_submission.csv',
    score_public=True,
)
print('final CSV:', final_csv)

## Evaluation handoff

The public score is a sanity check, not training. Read [08b_model_evaluation.ipynb](08b_model_evaluation.ipynb) for the compact disagreement diagrams and [09_experimental_results.ipynb](09_experimental_results.ipynb) only if the alternative-policy audit is needed. The organiser-facing explanation is [ORGANISER_REPORT.md](../ORGANISER_REPORT.md).

In [ ]:
scoreboard = pd.read_csv(ROOT / 'evaluation' / 'scoreboard.csv')
scoreboard.loc[scoreboard['submission'].eq('final_submission.csv'), [
    'submission', 'macro_f1', 'spell_f1_mean', 'trans_f1_25'
]]